In [9]:
import pandas as pd
import pathlib
import random
import numpy as np
import gc
random.seed(0)

def fast_integrity_check(xenium_base_path, chunks_base_path, sample_size=50):
    x_root = pathlib.Path(xenium_base_path)
    c_root = pathlib.Path(chunks_base_path)

    # all_parquets = list(x_root.glob("*/transcripts.parquet"))
    all_parquets = [pathlib.Path(f'{x}/transcripts.parquet') for x in ['/data/srlab/AMP_collab/data/early_disease_synovium//xenium/Xenium_RA-SYN-EDP2/Level_2/4078378347-02-01/',
    '/data/srlab/AMP_collab/data/early_disease_synovium//xenium/Xenium_RA-SYN-EDP2/Level_2/4078382737-02-01/',
    '/data/srlab/AMP_collab/data/early_disease_synovium//xenium/Xenium_RA-SYN-EDP2/Level_2/4078379894-02-01/',
    '/data/srlab/AMP_collab/data/early_disease_synovium//xenium/Xenium_RA-SYN-EDP2/Level_2/4078382892-02-01/']]
    if not all_parquets:
        print("No Parquet files found.")
        return

    sampled_paths = random.sample(all_parquets, min(len(all_parquets), sample_size))

    for p_path in sampled_paths:
        sid = p_path.parent.name
        chunk_dir = c_root / sid / "chunks"

        print(f"\n>>> Auditing SID: {sid}")

        # 1. Load Parquet: Use 'transcript_id' as index, create 'total_count' column
        # Optimization: We only load the ID column
        df = pd.read_parquet(p_path, columns=['transcript_id'])
        df.set_index('transcript_id', inplace=True)
        df['total_count'] = 0

        csv_files = list(chunk_dir.glob("*.csv"))
        if not csv_files:
            print(f"    No chunks found for {sid}")
            continue

        # 2. Process chunks and update the master DataFrame
        for i, csv_file in enumerate(csv_files):
            # Load only the ID column from the chunk
            # molecule_id is what you previously said the CSV header uses
            chunk_df = pd.read_csv(csv_file, usecols=['molecule_id'], dtype=str)

            counts = chunk_df['molecule_id'].value_counts()
            counts.index = counts.index.astype(np.uint64)

            # Update the master DF
            mapped_values = df.index.map(counts)

            print(f"Mapped values found (non-NaN): {mapped_values.notna().sum()}")

            df['total_count'] = df['total_count'].add(mapped_values.fillna(0))

            found_sofar = (df['total_count'] > 0).sum()

            print(f"Total IDs successfully marked in master DF: {found_sofar}")
            print(df.total_count.value_counts())

            print(f"    Processed chunk {i+1}/{len(csv_files)}: {csv_file.name}")
            del chunk_df
            del mapped_values
            gc.collect()

        # 3. Analyze Results
        print(f"\n    Analysis complete for {sid}.")

        missing = df[df['total_count'] == 0]
        duplicates = df[df['total_count'] > 1]

        if missing.empty and duplicates.empty:
            print("    ✅ Success: 1:1 Mapping confirmed.")
        else:
            if not missing.empty:
                print(f"    ❌ MISSING: {len(missing)} transcripts found in Parquet but 0 chunks.")
            if not duplicates.empty:
                print(f"    ❌ DUPLICATES: {len(duplicates)} transcripts found in more than 1 chunk.")
                print("    Example duplicate IDs and their counts:")
                print(duplicates['total_count'].head(5))

if __name__ == "__main__":
    # XENIUM_BASE = "/data/srlab/AMP_collab/data/early_disease_synovium/xenium/Xenium_CTRL-SYN-EDP1/Level_2/"
    # XENIUM_BASE = "/data/srlab/AMP_collab/data/early_disease_synovium/xenium/Xenium_RA-SYN-EDP1/Level_2/"
    # XENIUM_BASE = "/data/srlab/AMP_collab/data/early_disease_synovium/xenium/Xenium_RA-SYN-ARBITRATE/Level_2/"
    XENIUM_BASE = "/data/srlab/AMP_collab/data/early_disease_synovium/xenium/Xenium_RA-SYN-EDP2/Level_2/"
    # XENIUM_BASE = "/data/srlab/AMP_collab/data/early_disease_synovium/xenium/Xenium_CTRL-SYN-EDP1/Level_2/"
    CHUNKS_BASE = "/data/srlab/AMP_collab/lakshay-yakir/3.cell_segmentation/out/" # wherever the {sid}/chunks/ folder lives

    fast_integrity_check(XENIUM_BASE, CHUNKS_BASE)


>>> Auditing SID: 4078382892-02-01
Mapped values found (non-NaN): 1856302
Total IDs successfully marked in master DF: 1856302
total_count
0.0    48544663
1.0     1856302
Name: count, dtype: int64
    Processed chunk 1/38: F00033.csv
Mapped values found (non-NaN): 1111927
Total IDs successfully marked in master DF: 2968229
total_count
0.0    47432736
1.0     2968229
Name: count, dtype: int64
    Processed chunk 2/38: F00034.csv
Mapped values found (non-NaN): 1860705
Total IDs successfully marked in master DF: 4828934
total_count
0.0    45572031
1.0     4828934
Name: count, dtype: int64
    Processed chunk 3/38: F00016.csv
Mapped values found (non-NaN): 1621258
Total IDs successfully marked in master DF: 6450192
total_count
0.0    43950773
1.0     6450192
Name: count, dtype: int64
    Processed chunk 4/38: F00008.csv
Mapped values found (non-NaN): 1553863
Total IDs successfully marked in master DF: 8004055
total_count
0.0    42396910
1.0     8004055
Name: count, dtype: int64
    Process